# nesy-highway-driving — Part 1: pure-neural baseline

Train **PPO** (recommended) and **DQN** on `highway-env` with discrete meta-actions, evaluate both on the same held-out seeds, record videos, and write the study metrics — all mirrored to Google Drive.

Everything is driven by `configs/highway.yaml`; the `.py` modules are function-only libraries imported here.

## 1. Mount Drive + bootstrap the repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# First run on a fresh runtime: fetch the bootstrap script, then run it.
# (If the repo is already cloned, just run: !bash /content/nesy-highway-driving/bash/setup_colab.sh)
!git clone https://github.com/silvergjeka22/nesy-highway-driving.git /content/nesy-highway-driving || git -C /content/nesy-highway-driving pull --ff-only
!bash /content/nesy-highway-driving/bash/setup_colab.sh

In [ ]:
import sys
sys.path.insert(0, '/content/nesy-highway-driving')
%cd /content/nesy-highway-driving

## 2. Load the config

In [ ]:
from utils import load_config, drive_path, save_json

cfg = load_config('configs/highway.yaml')
# Quick smoke test? Uncomment to shrink the run:
# from utils import deep_update
# cfg = deep_update(cfg, {'ppo': {'total_timesteps': 5000}, 'dqn': {'total_timesteps': 5000}})
cfg['paths']

## 3. Train PPO and DQN (checkpoints mirror to Drive)

In [ ]:
from agents.baselines import train_ppo, train_dqn

ppo_model = train_ppo(cfg)   # -> <drive>/checkpoints/ppo.zip
dqn_model = train_dqn(cfg)   # -> <drive>/checkpoints/dqn.zip

## 4. Evaluate both on the same held-out seeds

In [ ]:
from eval.evaluate import evaluate, record_video

ppo_metrics = evaluate(ppo_model, cfg)
dqn_metrics = evaluate(dqn_model, cfg)

save_json(ppo_metrics, drive_path(cfg, 'metrics', 'ppo.json'))
save_json(dqn_metrics, drive_path(cfg, 'metrics', 'dqn.json'))

print('PPO:', ppo_metrics['summary'])
print('DQN:', dqn_metrics['summary'])

## 5. PPO vs DQN comparison table

In [ ]:
import pandas as pd

def row(name, m):
    s = m['summary']
    return {
        'algo': name,
        'crash_rate': round(s['crash_rate'], 3),
        'on_road_%': f"{s['on_road_pct']['mean']:.1f}±{s['on_road_pct']['std']:.1f}",
        'overtakes': f"{s['overtakes']['mean']:.2f}±{s['overtakes']['std']:.2f}",
        'return': f"{s['return']['mean']:.2f}±{s['return']['std']:.2f}",
        'length': f"{s['length']['mean']:.1f}±{s['length']['std']:.1f}",
    }

pd.DataFrame([row('PPO', ppo_metrics), row('DQN', dqn_metrics)])

## 6. Record videos (mirror to Drive)

In [ ]:
record_video(ppo_model, cfg, drive_path(cfg, 'videos', 'ppo.mp4'))
record_video(dqn_model, cfg, drive_path(cfg, 'videos', 'dqn.mp4'))

## 7. (Next) Part 2 — NeSy
Freeze these checkpoints; Part 2 wraps the frozen policy with `nesy/roadmap.py` (`predicates`, `safety_shield`) and re-runs this exact harness with an added per-rule violation column.